# Project01 — dữ liệu, KB/KG/RAG: kiểm tra nền

Phase 1 kiểm artifact, schema, một forward TRAIN và vài query mới; không train, không mở TEST cũ và chưa đánh giá project. Đây chưa phải implementation agent hoàn chỉnh. Phase 2 bổ sung đầy đủ source thuật toán trong notebook 01/02/03. Không tự sửa hay sinh lại notebook từ producer sau khi đã seal.

In [ ]:
from pathlib import Path
import json, sys, hashlib, gzip, time, warnings, unicodedata, re
REPO=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'FINAL_PROJECT01_EXECUTION_PLAN.md').exists())
PROJECT=REPO/'project01'
sys.path.insert(0,str(PROJECT/'scripts'))
from run import activate
CONTEXT=activate()
RUN_TRAINING=False
RUN_TEST=False
RUN_EVALUATION=False
print(CONTEXT['python'])


## 1. Kiểm nguồn và hash

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def checked(record):
    path=(REPO/record['path']).resolve()
    if not path.is_relative_to(REPO) or not path.is_file():
        raise ValueError('Missing/outside-workspace artifact')
    with path.open('rb') as stream:
        actual=hashlib.file_digest(stream,'sha256').hexdigest()
    if actual!=record['sha256']:
        raise ValueError('Artifact SHA mismatch: '+record['path'])
    return path

ASSETS=read_json(PROJECT/'datasets/assets.json')
SCHEMA=read_json(PROJECT/'datasets/profile_schema.json')
for record in [*ASSETS['model'].values(),ASSETS['rag_manifest'],*ASSETS['rag_files'].values()]:
    checked(record)
print('SHA checks PASS; features:',len(SCHEMA['feature_order']))


## 2. Kiểm mô hình đã huấn luyện trên mẫu TRAIN

Score là đầu ra phân loại nhãn khảo sát lịch sử; không gọi là xác suất mắc bệnh tương lai. `age` là mã nhóm tuổi, không phải số tuổi.

In [ ]:
import numpy as np
import pandas as pd
# Phase 1 loads the existing implementation for a foundation probe only.
# Phase 2 notebook 02 will include the complete serving implementation in cells.
from src.common.numpy_engine import NumpyNetwork
from src.common.tensor_adapters import TensorAdapter
config=read_json(checked(ASSETS['model']['config']))
metadata=read_json(checked(ASSETS['model']['checkpoint_metadata']))
preprocessing=read_json(checked(ASSETS['model']['preprocessing']))
adapter=TensorAdapter(preprocessing)
assert adapter.output_feature_names==SCHEMA['feature_order']
assert metadata['provenance']['preprocessing_sha256']==ASSETS['model']['preprocessing']['sha256']
assert metadata['sha256']==ASSETS['model']['checkpoint']['sha256']
assert config['dataset_id']=='ch2_cdc_diabetes' and config['plan']['input_shape']==[21]
with np.load(checked(ASSETS['model']['checkpoint']),allow_pickle=False) as archive:
    weights={name:archive[name].copy() for name in archive.files}
assert set(weights)==set(metadata['arrays'])
assert all(list(value.shape)==metadata['arrays'][name]['shape'] and str(value.dtype)==metadata['arrays'][name]['dtype'] and np.isfinite(value).all() for name,value in weights.items())
network=NumpyNetwork(config['plan'],weights)
demo=read_json(PROJECT/'datasets/demo_profiles.json')['profiles'][0]
tensor=adapter.transform(pd.DataFrame([demo['profile']],columns=SCHEMA['feature_order']))
logit=network.forward(tensor)
score=float(1/(1+np.exp(-np.clip(logit[0,0],-60,60))))
assert tensor.shape==(1,21) and np.isfinite(score)
MODEL_PROBE={'status':'PASS','profile_id':demo['id'],'input_origin':'matched TRAIN sample','input_shape':list(tensor.shape),'score':score,'positive_when':ASSETS['positive_when'],'class_names':SCHEMA['class_names'],'dense_layers':sum(s['kind']=='dense' for s in config['plan']['steps']),'training_steps':0,'historical_TEST_forward':0}
display(MODEL_PROBE)


## 3. Tải sparse/LSA, corpus và graph; query thật

Graph hiện là concept → chunk; không phải đồ thị quan hệ nhân quả y khoa. Pickle/joblib chỉ được nạp sau kiểm hash từ manifest tin cậy.

In [ ]:
import joblib
rag=read_json(checked(ASSETS['rag_manifest']))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    indexes={role:joblib.load(checked(ASSETS['rag_files'][filename])) for role,filename in rag['indexes']['medical'].items()}
INDEX_WARNINGS=sorted({str(w.message) for w in caught})
CHUNKS={}
with gzip.open(checked(ASSETS['rag_files'][rag['chunks_file']]),'rt',encoding='utf-8') as stream:
    for line in stream:
        row=json.loads(line)
        if row['domain']=='medical': CHUNKS[row['chunk_id']]=row
with gzip.open(checked(ASSETS['rag_files'][rag['graph_file']]),'rt',encoding='utf-8') as stream:
    GRAPH=json.load(stream)['medical']
assert len(CHUNKS)==762 and len(GRAPH)==62
assert all(set(index.doc_ids)==set(CHUNKS) for index in indexes.values())
assert all(set(node['chunk_ids'])<=set(CHUNKS) for node in GRAPH.values())
# Real searches against the medical corpus; no reference to old QA questions/labels.
query='physical activity diabetes'
hits={role:index.search(query,3) for role,index in indexes.items()}
assert all(len(rank)==3 and all(key in CHUNKS and np.isfinite(value) for key,value in rank) for rank in hits.values())
display({role:[{'chunk_id':key,'score':float(value),'excerpt':CHUNKS[key]['text'][:240]} for key,value in rank] for role,rank in hits.items()})
print('Index/corpus/graph ID parity PASS; sklearn warnings retained:',INDEX_WARNINGS)


## 4. Probe ngôn ngữ

Corpus tiếng Anh. So query tiếng Anh, tiếng Việt trực tiếp và ánh xạ hữu hạn; không gọi ánh xạ này là dịch máy.

In [ ]:
def normalize_phrase(text):
    text=unicodedata.normalize('NFKD',text.casefold()).replace('đ','d')
    return re.sub(r'[^a-z0-9 ]+',' ',' '.join(c for c in text if not unicodedata.combining(c))).strip()

GLOSSARY=read_json(PROJECT/'datasets/query_glossary.json')['mapping']
def map_bounded_vietnamese(question):
    normalized=normalize_phrase(question)
    found=[]
    for source,target in sorted(GLOSSARY.items(),key=lambda pair:-len(pair[0])):
        if re.search(r'\b'+re.escape(normalize_phrase(source))+r'\b',normalized):
            if target not in found:found.append(target)
    return ' '.join(found)

pairs=[('physical activity diabetes','vận động thể chất tiểu đường'),('kidney disease diabetes','bệnh thận tiểu đường'),('diabetes retinopathy','võng mạc tiểu đường')]
language_rows=[]
for english,vietnamese in pairs:
    mapped=map_bounded_vietnamese(vietnamese)
    ranks={label:indexes['sparse'].search(query,3) for label,query in [('en',english),('vi_raw',vietnamese),('vi_glossary',mapped)]}
    language_rows.append({'english':english,'vietnamese':vietnamese,'mapped':mapped,'top_ids':{k:[x[0] for x in v] for k,v in ranks.items()},'top_scores':{k:[float(x[1]) for x in v] for k,v in ranks.items()},'en_vs_raw_top3_overlap':len(set(x[0] for x in ranks['en'])&set(x[0] for x in ranks['vi_raw'])),'en_vs_glossary_top3_overlap':len(set(x[0] for x in ranks['en'])&set(x[0] for x in ranks['vi_glossary']))})
display(pd.DataFrame(language_rows))
print('Diagnostic only: top-ID overlap is not QA correctness or translation accuracy.')


## 5. Lưu evidence mới riêng project

In [ ]:
import importlib.metadata as metadata_module
report={'status':'PASS','model':MODEL_PROBE,'medical_chunks':len(CHUNKS),'medical_documents':len({r['document_id'] for r in CHUNKS.values()}),'graph_anchors':len(GRAPH),'index_corpus_id_parity':True,'graph_chunk_parity':True,'language_probes':language_rows,'index_warnings':INDEX_WARNINGS,'runtime':{name:metadata_module.version(name) for name in ['numpy','scipy','scikit-learn','joblib','pandas']},'python':sys.version,'training_steps':0,'historical_TEST_forward':0,'new_evaluation_runs':0}
path=PROJECT/'artifacts/phase1/probe.json'
path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(path)
